# 1. Train

Builds the search index and trains flan-t5-small. Use a T4 GPU and choose *Runtime > Run all*.
If the runtime stops, run all again: finished steps are reused and training continues from its last checkpoint.

In [ ]:
%pip install -q "transformers==5.22.2" "peft==0.21.2" accelerate sentencepiece protobuf pyarrow
%pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [ ]:
# True runs a quick check with 300 rows. Use the same value in both notebooks.
SMOKE_TEST = False

import json
import sys
import time

import pandas as pd
from google.colab import drive

NOTEBOOK_START = time.time()
drive.mount("/content/drive")
sys.dont_write_bytecode = True
sys.path.insert(0, "/content/drive/MyDrive/agricultural_qa")

import agri_common as ac

ac.require_gpu()
paths = ac.Paths(SMOKE_TEST)

Mounted at /content/drive
GPU: Tesla T4
Run folder: /content/drive/MyDrive/agricultural_qa/fast_v1/2effa812


## Load data

In [ ]:
with ac.stage("Load data", paths):
    data = ac.load_data(paths, SMOKE_TEST)
train_df, val_df, test_df = data["train"], data["val"], data["test"]
print(f"train {len(train_df)} | val {len(val_df)} | test {len(test_df)} | unseen {len(data['unseen'])}")


--- Load data ---


## Build the search index

In [ ]:
with ac.stage("Build index", paths):
    encoder = ac.Encoder()
    ac.build_index(train_df, encoder, paths)
    retriever = ac.Retriever(encoder, paths)

## Evaluate retrieval

Compares embedding search with hybrid search. A training record counts as relevant when its answer has cosine >= 0.80 with the test answer.
These scores can't be compared with the old notebook's results.

In [ ]:
import numpy as np

results_file = paths.outputs / "retrieval_results.csv"

with ac.stage("Evaluate retrieval", paths):
    if not results_file.exists():
        questions = test_df["question"].tolist()
        rankings = {
            "Baseline": [retriever.records(h) for h in retriever.search(questions, top_k=50, hybrid=False)],
            "Final": [retriever.records(h) for h in retriever.search(questions, top_k=50)],
        }
        train_vectors = encoder.encode(train_df["final_answer"], "passage: ", batch_size=64, max_length=512)
        gold_vectors = encoder.encode(test_df["final_answer"], "passage: ", batch_size=64, max_length=512)
        train_ids = train_df["question_id"].to_numpy()
        relevant = [set(train_ids[row >= 0.80]) for row in gold_vectors @ train_vectors.T]
        print("Test questions with a relevant record:", sum(map(bool, relevant)), "of", len(relevant))

        languages = test_df["language"].to_numpy()
        rows = []
        for name, ranked in rankings.items():
            for language in ("overall", "en", "hi", "pa"):
                keep = range(len(test_df)) if language == "overall" else np.flatnonzero(languages == language)
                if len(keep):
                    rows.append({"model": name, "language": language, "n": len(keep),
                                 **ac.retrieval_metrics([ranked[i] for i in keep], [relevant[i] for i in keep])})
        pd.DataFrame(rows).to_csv(results_file, index=False)

display(pd.read_csv(results_file).round(4))

## Build training prompts

Each training question is searched with its own answer excluded, so the model never sees the answer it has to write.

In [ ]:
train_file, val_file = paths.prompts / "train.parquet", paths.prompts / "val.parquet"


def make_prompts(table, exclude=None):
    hits = retriever.search(table["question"], exclude=exclude)
    if exclude is not None and any(h["row"] in own for row_hits, own in zip(hits, exclude) for h in row_hits):
        raise AssertionError("Leakage: a training prompt contains its own answer")
    prompts = [ac.make_prompt(q, c) for q, c in zip(table["question"], retriever.contexts(hits))]
    return pd.DataFrame({"prompt": prompts, "answer": table["final_answer"].to_numpy()})


with ac.stage("Build prompts", paths):
    if not (train_file.exists() and val_file.exists()):
        make_prompts(train_df, [retriever.own_rows(r) for r in train_df.itertuples()]).to_parquet(train_file, index=False)
        make_prompts(val_df.sample(n=min(300, len(val_df)), random_state=ac.SEED)).to_parquet(val_file, index=False)
    train_prompts, val_prompts = pd.read_parquet(train_file), pd.read_parquet(val_file)

print(len(train_prompts), "train prompts |", len(val_prompts), "val prompts\n")
print(train_prompts["prompt"].iloc[0][:800])

# The embedding model isn't needed for training.
del retriever, encoder
ac.free_gpu()

## Train

Training stops by itself when the time budget in `agri_common.py` runs out, and it keeps the best checkpoint.
If a finished model already exists, this step is skipped.

In [ ]:
import torch
from peft import LoraConfig, TaskType, get_peft_model
from transformers import (AutoModelForSeq2SeqLM, AutoTokenizer, DataCollatorForSeq2Seq, EarlyStoppingCallback,
                          Seq2SeqTrainer, Seq2SeqTrainingArguments, TrainerCallback)
from transformers.trainer_utils import get_last_checkpoint

info_file = paths.model / "training_info.json"


class TimeBudget(TrainerCallback):
    def __init__(self, minutes):
        self.deadline = time.time() + minutes * 60
        self.stopped = False

    def on_step_end(self, args, state, control, **kwargs):
        if time.time() > self.deadline and not self.stopped:
            print(f"Time budget reached at step {state.global_step}, stopping.")
            self.stopped = True
            control.should_evaluate = control.should_save = control.should_training_stop = True


def tokenize_examples(table):
    sources = tokenizer(table["prompt"].tolist(), truncation=True, max_length=ac.MAX_SOURCE_TOKENS)
    labels = tokenizer(table["answer"].tolist(), truncation=True, max_length=ac.MAX_TARGET_TOKENS)["input_ids"]
    return [{"input_ids": i, "attention_mask": m, "labels": l}
            for i, m, l in zip(sources["input_ids"], sources["attention_mask"], labels)]


if info_file.exists():
    print("Finished model found, skipping training.")
else:
    with ac.stage("Train", paths):
        tokenizer = AutoTokenizer.from_pretrained(ac.GEN_MODEL)
        model = AutoModelForSeq2SeqLM.from_pretrained(ac.GEN_MODEL, dtype=torch.float32)
        model.config.use_cache = False
        model = get_peft_model(model, LoraConfig(task_type=TaskType.SEQ_2_SEQ_LM, r=ac.LORA_R, lora_alpha=ac.LORA_ALPHA,
                                                 lora_dropout=0.05, target_modules=["q", "k", "v", "o"]))
        model.print_trainable_parameters()

        eval_steps = 5 if SMOKE_TEST else 200
        args = Seq2SeqTrainingArguments(
            output_dir=str(paths.checkpoints),
            num_train_epochs=1 if SMOKE_TEST else ac.EPOCHS,
            per_device_train_batch_size=ac.TRAIN_BATCH_SIZE,
            per_device_eval_batch_size=32,
            learning_rate=ac.LEARNING_RATE,
            lr_scheduler_type="cosine",
            warmup_steps=0.03,
            eval_strategy="steps",
            eval_steps=eval_steps,
            save_strategy="steps",
            save_steps=eval_steps,
            save_total_limit=2,
            load_best_model_at_end=True,
            metric_for_best_model="eval_loss",
            greater_is_better=False,
            logging_steps=25,
            report_to="none",
            remove_unused_columns=False,
            label_names=["labels"],
            seed=ac.SEED,
        )
        minutes_left = max(5, ac.TRAIN_BUDGET_MINUTES - (time.time() - NOTEBOOK_START) / 60)
        print(f"Training time budget: {minutes_left:.0f} minutes")
        budget = TimeBudget(minutes_left)
        trainer = Seq2SeqTrainer(
            model=model,
            args=args,
            train_dataset=tokenize_examples(train_prompts),
            eval_dataset=tokenize_examples(val_prompts),
            data_collator=DataCollatorForSeq2Seq(tokenizer, pad_to_multiple_of=8),
            callbacks=[EarlyStoppingCallback(early_stopping_patience=3), budget],
        )
        trainer.train(resume_from_checkpoint=get_last_checkpoint(str(paths.checkpoints)))
        trainer.model.save_pretrained(str(paths.model))
        pd.DataFrame(trainer.state.log_history).to_csv(paths.outputs / "training_history.csv", index=False)
        # Written last: its presence means the model is complete.
        info_file.write_text(json.dumps({
            "train_examples": len(train_prompts),
            "best_val_loss": trainer.state.best_metric,
            "steps": trainer.state.global_step,
            "stopped_by_time_budget": budget.stopped,
        }, indent=2))
        del trainer, model
        ac.free_gpu()

print(info_file.read_text())

## Loss curve

In [ ]:
import matplotlib.pyplot as plt

history = pd.read_csv(paths.outputs / "training_history.csv")
for column, label in (("loss", "Training loss"), ("eval_loss", "Validation loss")):
    if column in history:
        points = history.dropna(subset=[column])
        plt.plot(points["step"], points[column], label=label)
plt.xlabel("Training step")
plt.ylabel("Loss")
plt.legend()
plt.show()

## Finish

In [ ]:
print(f"Notebook time: {(time.time() - NOTEBOOK_START) / 60:.1f} minutes")
drive.flush_and_unmount()  # makes sure every file has reached Drive
print("Saved. Open 02_inference.ipynb in a fresh runtime.")